# Word Prediction Model (GPT-style Language Model)

A comprehensive tutorial on building a text generation model using RNN (simplified GPT approach).

---

## 📚 Table of Contents
1. [Introduction & Theory](#intro)
2. [Language Modeling vs Translation](#comparison)
3. [GPT Architecture Explained](#gpt)
4. [Implementation (RNN-based)](#implementation)
5. [Text Generation](#generation)
6. [Scaling to GPT](#scaling)
7. [References](#references)

---

## <a id='intro'></a>🎯 Introduction

### What is a Language Model?

A **language model** learns the probability distribution over sequences of words. Given previous words, it predicts the next word.

```python
Input:  "The cat sat on the"
Output: "mat"  (or "chair", "floor", etc.)
```

### Applications:

1. **Text Generation**: Write stories, articles, code
2. **Autocomplete**: Phone keyboards, IDE suggestions
3. **Chatbots**: GPT-3, ChatGPT, Claude
4. **Code Completion**: GitHub Copilot
5. **Text Summarization**: Condense long documents

### Key Concept: Auto-regressive Generation

The model generates text **one word at a time**, using its own previous predictions:

```
Step 1: "The" → predict → "cat"
Step 2: "The cat" → predict → "sat"
Step 3: "The cat sat" → predict → "on"
Step 4: "The cat sat on" → predict → "the"
Step 5: "The cat sat on the" → predict → "mat"
```

### Mathematical Formulation:

Given a sequence of words w₁, w₂, ..., wₙ, we want:

```
P(wₙ | w₁, w₂, ..., wₙ₋₁)
```

This is the probability of word wₙ given all previous words.

---

## <a id='comparison'></a>🔄 Language Modeling vs Translation

### Translation (Encoder-Decoder):

```
┌─────────────────────────────────────────────┐
│  TRANSLATION (what we built before)         │
└─────────────────────────────────────────────┘

Input:  "I am cold" (English)
         ↓
    ┌─────────┐
    │ ENCODER │ → Context Vector
    └─────────┘         ↓
                   ┌─────────┐
                   │ DECODER │
                   └─────────┘
                        ↓
Output: "j'ai froid" (French)

✓ Two languages
✓ Fixed-length context vector
✓ Encoder + Decoder
```

### Language Modeling (Decoder-only):

```
┌─────────────────────────────────────────────┐
│  LANGUAGE MODEL (GPT-style)                 │
└─────────────────────────────────────────────┘

Input:  "The cat sat on"
         ↓
    ┌─────────┐
    │ DECODER │ (only decoder, no encoder!)
    └─────────┘
         ↓
Output: "the"

Then: "The cat sat on the"
         ↓
    ┌─────────┐
    │ DECODER │
    └─────────┘
         ↓
Output: "mat"

✓ Same language
✓ No context vector bottleneck
✓ Decoder only (auto-regressive)
```

### Key Differences:

| Aspect | Translation | Language Model |
|--------|-------------|----------------|
| **Architecture** | Encoder + Decoder | Decoder only |
| **Input/Output** | Different languages | Same language |
| **Task** | Map source → target | Predict next word |
| **Training Data** | Parallel sentences | Raw text corpus |
| **Context** | Fixed vector | Growing sequence |
| **Example** | EN→FR translation | Text completion |

---

## <a id='gpt'></a>🤖 GPT Architecture Explained

### What is GPT?

**GPT** = **G**enerative **P**re-trained **T**ransformer

- **Generative**: Creates new text
- **Pre-trained**: Trained on massive text corpus first
- **Transformer**: Uses attention mechanism (not RNN)

### GPT Evolution:

```
GPT-1 (2018):  117M parameters
GPT-2 (2019):  1.5B parameters
GPT-3 (2020):  175B parameters
GPT-4 (2023):  ~1.7T parameters (estimated)
```

### GPT Architecture (Simplified):

```python
Input: "The cat sat on"
   ↓
[Tokenization]
   ↓
[Word Embeddings] → Dense vectors
   ↓
[Positional Encoding] → Add position info
   ↓
[Transformer Blocks] × N layers
   │
   ├─ Multi-Head Self-Attention
   ├─ Feed-Forward Network
   └─ Layer Normalization
   ↓
[Linear Layer] → Vocabulary size
   ↓
[Softmax] → Probabilities
   ↓
Output: "the" (most likely next word)
```

### What We'll Build:

We'll build a **simplified version** using **RNN instead of Transformer**:

```
Input: "The cat sat on"
   ↓
[Word Embeddings]
   ↓
[LSTM/GRU] (instead of Transformer)
   ↓
[Linear Layer]
   ↓
[Softmax]
   ↓
Output: "the"
```

**Why RNN first?**
- Easier to understand
- Same core concepts
- Less computational resources
- Foundation for understanding Transformers

### Causal (Autoregressive) Masking:

**Important**: The model can only look at **previous words**, not future words!

```python
Training sequence: "The cat sat on the mat"

Step 1: Input="The"          → Predict="cat"
Step 2: Input="The cat"      → Predict="sat"
Step 3: Input="The cat sat"  → Predict="on"
...

✓ Can see: "The", "cat", "sat" (past)
✗ Cannot see: "the", "mat" (future)
```

This is called **causal** or **autoregressive** modeling.

---

## <a id='implementation'></a>💻 Implementation: RNN Language Model

Let's build a character-level and word-level language model using LSTM!

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import Counter
import re

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Prepare Training Data

### Training Corpus

For language modeling, we need:
- **Raw text**: Any text corpus (books, articles, code)
- **No parallel data**: Unlike translation!
- **Large corpus**: More data = better model

We'll use a simple story corpus for demonstration.

In [ ]:
# Sample training corpus (in practice, use much larger datasets)
corpus = """
The quick brown fox jumps over the lazy dog.
A journey of a thousand miles begins with a single step.
To be or not to be, that is the question.
All that glitters is not gold.
Where there is a will, there is a way.
Actions speak louder than words.
Practice makes perfect.
The early bird catches the worm.
Better late than never.
A picture is worth a thousand words.
When in Rome, do as the Romans do.
The pen is mightier than the sword.
Beauty is in the eye of the beholder.
Time flies when you are having fun.
You cannot judge a book by its cover.
Every cloud has a silver lining.
Fortune favors the bold.
The apple does not fall far from the tree.
Hope for the best, prepare for the worst.
Knowledge is power.
"""

# Preprocess: lowercase and clean
corpus = corpus.lower().strip()
corpus = re.sub(r'[^a-z\s.,!?]', '', corpus)  # Keep basic punctuation

print(f"Corpus length: {len(corpus)} characters")
print(f"\nFirst 200 characters:")
print(corpus[:200])

## 2. Build Vocabulary

We'll build **word-level** vocabulary (predicting words, not characters).

For GPT-style models, you'd use **subword tokenization** (BPE, WordPiece).

In [ ]:
# Tokenize into words
words = corpus.split()

print(f"Total words: {len(words)}")
print(f"Sample words: {words[:20]}")

# Build vocabulary
word_counts = Counter(words)
vocab = sorted(word_counts.keys())  # Alphabetically sorted

# Create mappings
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

vocab_size = len(vocab)

print(f"\nVocabulary size: {vocab_size}")
print(f"\nMost common words:")
for word, count in word_counts.most_common(10):
    print(f"  '{word}': {count} times")

print(f"\nSample vocabulary: {vocab[:20]}")

## 3. Create Training Sequences

### Sequence Creation Strategy:

Given text: `"The cat sat on the mat"`

Create training pairs:
```python
Input: ["The"]                    → Target: "cat"
Input: ["The", "cat"]            → Target: "sat"
Input: ["The", "cat", "sat"]    → Target: "on"
Input: ["The", "cat", "sat", "on"] → Target: "the"
...
```

Or use **fixed-length sequences** (sliding window):
```python
Sequence length = 3
Input: ["The", "cat", "sat"]    → Target: "on"
Input: ["cat", "sat", "on"]     → Target: "the"
Input: ["sat", "on", "the"]     → Target: "mat"
```

In [ ]:
def create_sequences(words, word2idx, seq_length=5):
    """
    Create training sequences using sliding window.
    
    Args:
        words: List of words
        word2idx: Word to index mapping
        seq_length: Length of input sequence
    
    Returns:
        X: Input sequences (indices)
        y: Target words (indices)
    """
    X = []
    y = []
    
    # Sliding window approach
    for i in range(len(words) - seq_length):
        # Input: seq_length words
        input_seq = words[i:i+seq_length]
        # Target: next word
        target_word = words[i+seq_length]
        
        # Convert to indices
        input_indices = [word2idx[word] for word in input_seq]
        target_idx = word2idx[target_word]
        
        X.append(input_indices)
        y.append(target_idx)
    
    return np.array(X), np.array(y)

# Create sequences
SEQ_LENGTH = 5  # Use previous 5 words to predict next word
X, y = create_sequences(words, word2idx, SEQ_LENGTH)

print(f"Number of training sequences: {len(X)}")
print(f"Input shape: {X.shape}  # (num_sequences, seq_length)")
print(f"Target shape: {y.shape}  # (num_sequences,)")

# Show examples
print(f"\nTraining Examples:")
print("="*70)
for i in range(5):
    input_words = [idx2word[idx] for idx in X[i]]
    target_word = idx2word[y[i]]
    print(f"Input: {' '.join(input_words):40} → Target: {target_word}")

## 4. Define Language Model (RNN)

### Architecture:

```python
Input sequence: [w1, w2, w3, w4, w5]
    ↓
[Embedding Layer] → Convert indices to vectors
    ↓
[LSTM/GRU] → Process sequence
    ↓
[Take last hidden state] → Summary of sequence
    ↓
[Linear Layer] → Project to vocabulary size
    ↓
[Softmax] → Probability distribution over words
    ↓
Output: Probability for each word in vocabulary
```

### Differences from Translation Decoder:

- **No encoder**: We don't have a source language
- **No context vector**: We directly process input sequence
- **Predict from vocabulary**: Output is next word in same language

In [ ]:
class LanguageModelRNN(nn.Module):
    """
    RNN-based Language Model for word prediction.
    
    This is a simplified version of GPT (using RNN instead of Transformer).
    
    Architecture:
        Input → Embedding → LSTM → Linear → Softmax
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.3):
        super(LanguageModelRNN, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # LSTM (can also use GRU)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # Output layer: project to vocabulary
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        """
        Forward pass.
        
        Args:
            x: (batch_size, seq_length) - input word indices
            hidden: Optional hidden state from previous step
        
        Returns:
            output: (batch_size, vocab_size) - scores for each word
            hidden: Hidden state for next step
        """
        # Embed words
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        embedded = self.dropout(embedded)
        
        # Process through LSTM
        if hidden is None:
            lstm_out, hidden = self.lstm(embedded)
        else:
            lstm_out, hidden = self.lstm(embedded, hidden)
        
        # Take output from last time step
        # lstm_out shape: (batch, seq_len, hidden_dim)
        last_output = lstm_out[:, -1, :]  # (batch, hidden_dim)
        
        # Apply dropout
        last_output = self.dropout(last_output)
        
        # Project to vocabulary size
        output = self.fc(last_output)  # (batch, vocab_size)
        
        return output, hidden
    
    def init_hidden(self, batch_size):
        """Initialize hidden state"""
        weight = next(self.parameters())
        return (weight.new_zeros(self.num_layers, batch_size, self.hidden_dim),
                weight.new_zeros(self.num_layers, batch_size, self.hidden_dim))

# Initialize model
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3

model = LanguageModelRNN(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

print("Language Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# Parameter breakdown
print("\nParameter breakdown:")
for name, param in model.named_parameters():
    print(f"  {name:20} {param.shape} = {param.numel():,} params")

## 5. Training Function

### Training Process:

1. **Forward pass**: Input sequence → Predict next word
2. **Calculate loss**: Compare prediction with ground truth
3. **Backpropagation**: Update weights
4. **Repeat**: Iterate through entire corpus

### Loss Function:

**CrossEntropyLoss**: Measures how well predicted distribution matches actual next word.

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Convert to tensors
X_train = torch.LongTensor(X).to(device)
y_train = torch.LongTensor(y).to(device)

# Training parameters
EPOCHS = 500
BATCH_SIZE = 32
PRINT_EVERY = 50

print("Training Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: 0.001")
print(f"  Training samples: {len(X_train)}")
print("\nStarting training...\n")
print("="*70)

losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    
    # Shuffle data
    indices = torch.randperm(len(X_train))
    
    epoch_loss = 0
    num_batches = 0
    
    # Mini-batch training
    for i in range(0, len(X_train), BATCH_SIZE):
        # Get batch
        batch_indices = indices[i:i+BATCH_SIZE]
        batch_X = X_train[batch_indices]
        batch_y = y_train[batch_indices]
        
        # Forward pass
        optimizer.zero_grad()
        output, _ = model(batch_X)
        
        # Calculate loss
        loss = criterion(output, batch_y)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping (prevent exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        # Update weights
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    
    # Print progress
    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f}")
        
        # Generate sample text
        model.eval()
        with torch.no_grad():
            # Use first sequence as seed
            seed = X_train[0:1]
            output, _ = model(seed)
            _, predicted = torch.max(output, 1)
            
            seed_words = [idx2word[idx.item()] for idx in seed[0]]
            predicted_word = idx2word[predicted.item()]
            
            print(f"  Sample: '{' '.join(seed_words)}' → '{predicted_word}'")
        print("-" * 70)

print("\n✓ Training complete!")

## 6. Plot Training Loss

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(losses, alpha=0.6, color='blue')
if len(losses) >= 20:
    window = 20
    moving_avg = np.convolve(losses, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(losses)), moving_avg, color='red', linewidth=2, label='Moving Avg')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(losses, alpha=0.6, color='blue')
if len(losses) >= 20:
    plt.plot(range(window-1, len(losses)), moving_avg, color='red', linewidth=2, label='Moving Avg')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('Training Loss (Log Scale)')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Improvement: {((losses[0] - losses[-1])/losses[0]*100):.1f}%")

## 7. Evaluate Model

### Evaluation Metrics:

1. **Perplexity**: Lower is better (measures prediction uncertainty)
   - `Perplexity = exp(loss)`
   - Perfect model: Perplexity = 1
   - Random guessing: Perplexity = Vocabulary size

2. **Accuracy**: Percentage of correct next-word predictions

In [ ]:
# Evaluate on training data
model.eval()

correct = 0
total = 0

print("Evaluation Results:\n")
print("="*90)

with torch.no_grad():
    for i in range(min(20, len(X_train))):
        input_seq = X_train[i:i+1]
        target = y_train[i]
        
        # Predict
        output, _ = model(input_seq)
        _, predicted = torch.max(output, 1)
        
        # Convert to words
        input_words = [idx2word[idx.item()] for idx in input_seq[0]]
        target_word = idx2word[target.item()]
        predicted_word = idx2word[predicted.item()]
        
        is_correct = predicted.item() == target.item()
        if is_correct:
            correct += 1
        total += 1
        
        status = "✓" if is_correct else "✗"
        print(f"{status} Input: {' '.join(input_words):35} | Target: {target_word:10} | Predicted: {predicted_word}")

print("="*90)
print(f"\nAccuracy (sample): {correct}/{total} ({correct/total*100:.1f}%)")

# Calculate perplexity
perplexity = np.exp(losses[-1])
print(f"Perplexity: {perplexity:.2f}")
print(f"\n(Lower perplexity = better model. Random guessing ≈ {vocab_size})")

## <a id='generation'></a>8. Text Generation

### Generation Strategies:

1. **Greedy Decoding**: Always pick most likely word
   - Fast but repetitive

2. **Sampling**: Sample from probability distribution
   - More diverse but can be incoherent

3. **Top-K Sampling**: Sample from top K most likely words
   - Balance between diversity and quality

4. **Temperature Sampling**: Control randomness
   - High temp = more random
   - Low temp = more conservative

In [ ]:
def generate_text(model, seed_text, word2idx, idx2word, num_words=20, 
                  temperature=1.0, top_k=None):
    """
    Generate text using the trained language model.
    
    Args:
        model: Trained language model
        seed_text: Initial text to start generation
        word2idx, idx2word: Vocabulary mappings
        num_words: Number of words to generate
        temperature: Sampling temperature (higher = more random)
        top_k: If set, only sample from top k words
    
    Returns:
        Generated text
    """
    model.eval()
    
    # Prepare seed
    words = seed_text.lower().split()
    
    # Ensure we have at least SEQ_LENGTH words
    if len(words) < SEQ_LENGTH:
        print(f"Warning: Seed text too short. Need at least {SEQ_LENGTH} words.")
        return seed_text
    
    generated_words = words.copy()
    
    with torch.no_grad():
        for _ in range(num_words):
            # Take last SEQ_LENGTH words as input
            input_words = generated_words[-SEQ_LENGTH:]
            
            # Convert to indices (handle unknown words)
            try:
                input_indices = [word2idx[w] for w in input_words]
            except KeyError:
                print(f"Unknown word in seed. Using available words.")
                break
            
            # Prepare input tensor
            input_tensor = torch.LongTensor([input_indices]).to(device)
            
            # Forward pass
            output, _ = model(input_tensor)
            
            # Apply temperature
            output = output / temperature
            
            # Apply softmax to get probabilities
            probs = torch.softmax(output, dim=1).cpu().numpy()[0]
            
            # Top-K sampling
            if top_k is not None:
                top_indices = np.argsort(probs)[-top_k:]
                top_probs = probs[top_indices]
                top_probs = top_probs / top_probs.sum()  # Renormalize
                next_idx = np.random.choice(top_indices, p=top_probs)
            else:
                # Sample from full distribution
                next_idx = np.random.choice(len(probs), p=probs)
            
            # Get next word
            next_word = idx2word[next_idx]
            generated_words.append(next_word)
    
    return ' '.join(generated_words)

# Test text generation with different strategies
seed = "the quick brown fox jumps"

print("Text Generation Examples:\n")
print("="*70)
print(f"Seed text: '{seed}'\n")

# Greedy (temperature = 0.5, deterministic-ish)
print("1. Conservative (temperature=0.5):")
text1 = generate_text(model, seed, word2idx, idx2word, num_words=15, temperature=0.5)
print(f"   {text1}\n")

# Balanced (temperature = 1.0)
print("2. Balanced (temperature=1.0):")
text2 = generate_text(model, seed, word2idx, idx2word, num_words=15, temperature=1.0)
print(f"   {text2}\n")

# Creative (temperature = 1.5)
print("3. Creative (temperature=1.5):")
text3 = generate_text(model, seed, word2idx, idx2word, num_words=15, temperature=1.5)
print(f"   {text3}\n")

# Top-K sampling
print("4. Top-K sampling (k=10):")
text4 = generate_text(model, seed, word2idx, idx2word, num_words=15, temperature=1.0, top_k=10)
print(f"   {text4}\n")

print("="*70)

## 9. Interactive Text Generation

In [ ]:
def complete_text(seed_text, num_words=10):
    """Interactive text completion"""
    print(f"\nInput: {seed_text}")
    generated = generate_text(model, seed_text, word2idx, idx2word, 
                             num_words=num_words, temperature=1.0, top_k=10)
    print(f"Output: {generated}\n")
    return generated

# Try different seed texts
print("Interactive Text Completion:\n")
print("="*70)

complete_text("the early bird catches the", 5)
complete_text("all that glitters is not", 5)
complete_text("actions speak louder than", 5)
complete_text("a picture is worth a", 5)

print("="*70)
print("\nTry your own: complete_text('your seed text here', 10)")

## 10. Save Model

In [ ]:
import os

# Save model
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab': vocab,
    'word2idx': word2idx,
    'idx2word': idx2word,
    'hyperparameters': {
        'vocab_size': vocab_size,
        'embedding_dim': EMBEDDING_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
        'seq_length': SEQ_LENGTH
    },
    'loss_history': losses
}

save_path = 'language_model_rnn.pth'
torch.save(checkpoint, save_path)
print(f"✓ Model saved as '{save_path}'")
print(f"  File size: {os.path.getsize(save_path) / 1024:.1f} KB")

---

## <a id='scaling'></a>🚀 Scaling to GPT-like Models

### What We Built vs. GPT:

| Component | Our RNN Model | GPT (Transformer) |
|-----------|---------------|-------------------|
| **Architecture** | LSTM/GRU | Transformer (Self-Attention) |
| **Parameters** | ~100K-1M | 175B (GPT-3) |
| **Context Length** | 5-10 words | 2048-32k tokens |
| **Training Data** | Small corpus | Hundreds of GB |
| **Tokenization** | Word-level | Subword (BPE) |
| **Training Time** | Minutes | Months on supercomputers |

### How to Scale Up:

#### 1. **Use Transformer Architecture** ⭐

Replace LSTM with Transformer blocks:

```python
# Instead of:
self.lstm = nn.LSTM(...)

# Use:
self.transformer = nn.TransformerDecoder(
    nn.TransformerDecoderLayer(
        d_model=512,
        nhead=8,
        dim_feedforward=2048
    ),
    num_layers=12
)
```

**Why Transformer?**
- **Parallel processing**: RNN is sequential (slow)
- **Long-range dependencies**: Attention mechanism
- **Scalability**: Transformers scale better with data/compute

#### 2. **Increase Model Size**

```python
# GPT-2 Small (117M parameters)
n_layers = 12
n_heads = 12
d_model = 768

# GPT-2 Large (774M parameters)
n_layers = 36
n_heads = 20
d_model = 1280

# GPT-3 (175B parameters)
n_layers = 96
n_heads = 96
d_model = 12288
```

#### 3. **Use Subword Tokenization**

Instead of word-level, use **BPE** (Byte-Pair Encoding):

```python
# Word-level (our approach)
"unhappiness" → ["unhappiness"]  # OOV if not in vocab!

# BPE (GPT approach)
"unhappiness" → ["un", "happiness"]  # Subwords
"happiness" → ["happy", "ness"]
```

**Implementation:**
```python
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
```

#### 4. **Train on Massive Datasets**

```python
# Our corpus: ~500 words

# GPT-3 training data:
# - Common Crawl: 410 billion tokens
# - WebText2: 19 billion tokens
# - Books: 67 billion tokens
# - Wikipedia: 3 billion tokens
# Total: ~500 billion tokens!
```

**Where to get data:**
- [The Pile](https://pile.eleuther.ai/): 825 GB open-source dataset
- [C4](https://www.tensorflow.org/datasets/catalog/c4): Colossal Clean Crawled Corpus
- [BookCorpus](https://huggingface.co/datasets/bookcorpus): 11,000 books

#### 5. **Implement Advanced Training Techniques**

```python
# Mixed precision training (faster)
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

# Distributed training (multiple GPUs)
torch.distributed.init_process_group(backend='nccl')

# Gradient accumulation (larger effective batch size)
accumulation_steps = 4

# Learning rate warmup and decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)
```

#### 6. **Use Pre-trained Models**

Instead of training from scratch, **fine-tune** existing models:

```python
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer

# Load pre-trained GPT-2
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Fine-tune on your data
trainer = Trainer(
    model=model,
    train_dataset=your_dataset,
    ...
)
trainer.train()
```

### Practical Implementation:

```python
# Install transformers
# pip install transformers

from transformers import pipeline

# Use GPT-2 for text generation
generator = pipeline('text-generation', model='gpt2')
text = generator(
    "The quick brown fox",
    max_length=50,
    num_return_sequences=3
)
print(text)
```

### Training GPT from Scratch:

**Resources needed:**
- GPT-2 Small: 1-4 GPUs, days
- GPT-2 Medium: 8-16 GPUs, weeks
- GPT-3: Thousands of GPUs, months, $5M+ in compute

**Recommended path:**
1. Start with our RNN model (understanding)
2. Implement small Transformer (learning)
3. Fine-tune GPT-2 (practical applications)
4. Use GPT-3/GPT-4 API (production)

---

## 📚 Summary

### What We Built:

1. ✅ **RNN Language Model**
   - Word-level prediction
   - LSTM-based architecture
   - Auto-regressive generation

2. ✅ **Text Generation**
   - Temperature sampling
   - Top-K sampling
   - Interactive completion

3. ✅ **Understanding Foundation**
   - Language modeling concepts
   - Difference from translation
   - Path to GPT-style models

### Key Concepts:

- **Language Model**: P(next_word | previous_words)
- **Auto-regressive**: Generate one word at a time
- **Decoder-only**: No encoder needed
- **Causal masking**: Can't see future words
- **Temperature**: Control randomness
- **Perplexity**: Evaluation metric

### Limitations:

1. **Small corpus**: Limited vocabulary and patterns
2. **RNN bottleneck**: Can't capture long-range dependencies well
3. **Sequential processing**: Slow compared to Transformers
4. **Word-level**: Can't handle unknown words gracefully

### Next Steps:

1. **Implement Transformer**: Replace LSTM with self-attention
2. **Use larger dataset**: Download Wikipedia, books, etc.
3. **Subword tokenization**: Implement BPE
4. **Fine-tune GPT-2**: Use Hugging Face transformers
5. **Explore GPT-3 API**: OpenAI API for production

---

## <a id='references'></a>📖 References & Resources

### Foundational Papers:

1. **Improving Language Understanding by Generative Pre-Training (GPT-1)**
   - Authors: Radford et al. (OpenAI, 2018)
   - Paper: https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf
   - 📌 **First GPT paper** - introduced pre-training + fine-tuning

2. **Language Models are Unsupervised Multitask Learners (GPT-2)**
   - Authors: Radford et al. (OpenAI, 2019)
   - Paper: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
   - 📌 **GPT-2** - showed zero-shot learning capabilities

3. **Language Models are Few-Shot Learners (GPT-3)**
   - Authors: Brown et al. (OpenAI, 2020)
   - Paper: https://arxiv.org/abs/2005.14165
   - 📌 **GPT-3** - 175B parameters, few-shot learning

4. **Attention Is All You Need (Transformer)**
   - Authors: Vaswani et al. (Google, 2017)
   - Paper: https://arxiv.org/abs/1706.03762
   - 📌 **Foundation of GPT** - Transformer architecture

### Tutorials & Courses:

5. **The Illustrated GPT-2**
   - URL: http://jalammar.github.io/illustrated-gpt2/
   - 📌 **Best visual explanation** of how GPT works

6. **Stanford CS224N: NLP with Deep Learning**
   - URL: http://web.stanford.edu/class/cs224n/
   - Lectures on language models and Transformers

7. **Hugging Face Course**
   - URL: https://huggingface.co/course
   - Practical guide to using Transformers library

8. **Andrej Karpathy's "makemore"**
   - URL: https://github.com/karpathy/makemore
   - Build GPT from scratch (character-level)

### Code Resources:

9. **nanoGPT**
   - URL: https://github.com/karpathy/nanoGPT
   - Minimal GPT implementation by Andrej Karpathy

10. **Hugging Face Transformers**
    - URL: https://github.com/huggingface/transformers
    - Pre-trained models and easy fine-tuning

11. **minGPT**
    - URL: https://github.com/karpathy/minGPT
    - Educational GPT implementation

### Datasets:

12. **The Pile**
    - URL: https://pile.eleuther.ai/
    - 825 GB diverse text dataset

13. **OpenWebText**
    - URL: https://huggingface.co/datasets/openwebtext
    - Open recreation of GPT-2's WebText

14. **C4 (Colossal Clean Crawled Corpus)**
    - URL: https://www.tensorflow.org/datasets/catalog/c4
    - Cleaned Common Crawl data

### Tools & APIs:

15. **OpenAI API**
    - URL: https://platform.openai.com/docs/
    - GPT-3/GPT-4 API for production use

16. **Cohere API**
    - URL: https://cohere.ai/
    - Alternative language model API

17. **Anthropic Claude**
    - URL: https://www.anthropic.com/
    - Constitutional AI approach

### Books & Articles:

18. **Speech and Language Processing**
    - URL: https://web.stanford.edu/~jurafsky/slp3/
    - Chapter 3: N-gram Language Models
    - Chapter 7: Neural Networks and Neural Language Models

19. **The Annotated Transformer**
    - URL: http://nlp.seas.harvard.edu/annotated-transformer/
    - Line-by-line Transformer implementation

### Concepts:

20. **Tokenization (BPE)**
    - Paper: https://arxiv.org/abs/1508.07909
    - Subword tokenization used in GPT

21. **Temperature in Sampling**
    - Blog: https://lukesalamone.github.io/posts/what-is-temperature/

22. **Top-K and Top-P (Nucleus) Sampling**
    - Paper: https://arxiv.org/abs/1904.09751
    - Better text generation strategies

23. **Perplexity Explained**
    - URL: https://towardsdatascience.com/perplexity-in-language-models-87a196019a94

### Video Tutorials:

24. **Andrej Karpathy: Let's build GPT**
    - URL: https://www.youtube.com/watch?v=kCc8FmEb1nY
    - 📌 **Best video tutorial** - build GPT from scratch

25. **3Blue1Brown: Visualizing Attention**
    - URL: https://www.youtube.com/watch?v=eMlx5fFNoYc
    - Visual explanation of Transformers

---

## 💡 Next Steps:

1. **Implement Transformer**: Replace LSTM with self-attention
2. **Train on larger corpus**: Download Wikipedia dump
3. **Implement BPE tokenization**: Handle rare words better
4. **Fine-tune GPT-2**: Use Hugging Face library
5. **Build a chatbot**: Add dialogue capabilities
6. **Watch Karpathy's videos**: Deep understanding of GPT

---

## 🎯 Comparison Summary:

| Task | Architecture | Use Case |
|------|-------------|----------|
| **Translation** | Encoder-Decoder | EN→FR, summarization |
| **Language Model (GPT)** | Decoder-only | Text generation, completion |
| **Understanding (BERT)** | Encoder-only | Classification, Q&A |

You now understand the foundation of **GPT**! 🚀

---